In [ ]:
import os
import sys
import math
import logging
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
from pixell import reproject, lensing, enmap, utils

sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping, TerminateOnNaN
from tensorflow.keras.optimizers import AdamW, Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import (
    HealpyChebyshev,
    HealpyPool,
    HealpyPseudoConv_Transpose,
)

from mlpng import Core
from mlpng.utils import setup_logging, RMSELoss, rmse_metrics
from mlpng.utils.dataloaders import KappaDataset, MapDataset
from mlpng.scn_jorik import get_model as get_fnl_model

# Setup logging for notebook
setup_logging("mlpng.notebook", level=logging.DEBUG)
logger = logging.getLogger("mlpng.notebook")

In [ ]:
print("Conda environment:", os.environ["CONDA_DEFAULT_ENV"])
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs: {len(gpus)}")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
core = Core(
    [
        "settings/n64.json",
        "--nsims",
        "10000",
        "--phi_scale",
        "1",
        "--shapes",
        "local",
        "--fnl_range",
        "-1000",
        "1000",
    ]
)
shapes = core.shapes

In [ ]:
batch_size = 32
max_epochs = 100
initial_LR = 5e-4

data_fraction = 0.1
unet_split = np.array([0.8, 0.1, 0.1]) * data_fraction
unet_duplicates = [25, 10, 2]

final_split = np.array([0.8, 0.1, 0.1]) * data_fraction
final_duplicates = [1, 1, 1]

strategy = tf.distribute.MirroredStrategy()

In [ ]:
date_time = tf.timestamp().numpy().astype(int)
run_name = f"slim-notebook-{date_time}"
save_dir = f"{core.dirs['model']}/{core.name}"
run_info = f"{core.shapes_str()}-{run_name}"

unet_keras_file = f"{save_dir}/unet-{run_info}.keras"
fnl_keras_file = f"{save_dir}/fnl-{run_info}.keras"

unet_cache = f"{core.name}/unet-{core.name}-f{data_fraction}"

os.makedirs(save_dir, exist_ok=True)
print(f"Run name: {run_name}")
print(f"U-Net file: {unet_keras_file}")

In [ ]:
callbacks = [
    TerminateOnNaN(),
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
]

In [ ]:
@tf.keras.saving.register_keras_serializable()
class EncoderBlock(tf.keras.layers.Layer):
    def __init__(
        self,
        nside,
        npix,
        fin,
        fout,
        activation,
        max_batch_size,
        K,
        use_bn=True,
        use_bias=False,
        pool=True,
        dropout_rate=0.1,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.nside = nside
        self.npix = npix
        self.fin = fin
        self.fout = fout
        self.activation_fn = activation
        self.max_batch_size = max_batch_size
        self.K = K
        self.use_bn = use_bn
        self.use_bias = use_bias
        self.dropout_rate = dropout_rate
        self.pool = pool

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "nside": self.nside,
                "npix": self.npix,
                "fin": self.fin,
                "fout": self.fout,
                "activation": self.activation_fn,
                "max_batch_size": self.max_batch_size,
                "K": self.K,
                "use_bn": self.use_bn,
                "use_bias": self.use_bias,
                "pool": self.pool,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config

    def build_layers(self):
        """Build and return the layer list for this encoder block."""
        layers = [
            HealpyChebyshev(
                K=self.K,
                Fout=self.fout,
                activation=self.activation_fn,
                use_bn=self.use_bn,
                use_bias=self.use_bias,
            )
        ]
        if self.dropout_rate > 0.0:
            layers.append(Dropout(self.dropout_rate))
        return layers

    def get_pool_layer(self):
        """Get pooling layer if enabled."""
        if self.pool:
            return [HealpyPool(1, "AVG")]
        return []

In [ ]:
@tf.keras.saving.register_keras_serializable()
class DecoderBlock(tf.keras.layers.Layer):
    def __init__(
        self,
        nside,
        npix,
        fin,
        fout,
        activation,
        K,
        max_batch_size,
        upsample=True,
        use_bn=True,
        use_bias=False,
        dropout_rate=0.1,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.nside = nside
        self.npix = npix
        self.fin = fin
        self.fout = fout
        self.activation_fn = activation
        self.max_batch_size = max_batch_size
        self.K = K
        self.use_bn = use_bn
        self.use_bias = use_bias
        self.dropout_rate = dropout_rate
        self.upsample = upsample

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "nside": self.nside,
                "npix": self.npix,
                "fin": self.fin,
                "fout": self.fout,
                "activation": self.activation_fn,
                "K": self.K,
                "max_batch_size": self.max_batch_size,
                "upsample": self.upsample,
                "use_bn": self.use_bn,
                "use_bias": self.use_bias,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config

    def build_layers(self):
        """Build and return the layer list for this decoder block."""
        layers = []
        if self.upsample:
            layers.append(HealpyPseudoConv_Transpose(1, self.fout))
        layers.extend(
            [
                HealpyChebyshev(
                    K=self.K,
                    Fout=self.fout,
                    activation=self.activation_fn,
                    use_bn=self.use_bn,
                    use_bias=self.use_bias,
                )
            ]
        )
        if self.dropout_rate > 0.0:
            layers.append(Dropout(self.dropout_rate))
        return layers

In [ ]:
@tf.keras.saving.register_keras_serializable()
class SimpleHealpyUNet:
    """U-Net without skip connections, using a single unified HealpyGCNN."""

    def __init__(
        self,
        input_shape,
        activation="relu",
        max_batch_size=32,
        dropout_rate=0.1,
        use_bn=True,
        use_bias=True,
    ):
        self.input_shape = input_shape
        self.activation = activation
        self.max_batch_size = max_batch_size
        self.npix = input_shape[1]
        self.nside = hp.npix2nside(self.npix)
        self.npol = input_shape[2]
        self.dropout_rate = dropout_rate
        self.use_bn = use_bn
        self.use_bias = use_bias

    def get_config(self):
        return {
            "input_shape": self.input_shape,
            "activation": self.activation,
            "max_batch_size": self.max_batch_size,
            "dropout_rate": self.dropout_rate,
            "use_bn": self.use_bn,
            "use_bias": self.use_bias,
        }

    def get_model(self):
        depth = 3  # int(math.log2(self.nside))
        base_channels = [self.npol] + [2 ** (i + 4) for i in range(depth + 1)]
        level_npixels = [self.npix // (4**i) for i in range(depth + 1)]
        level_nsides = [hp.npix2nside(npix) for npix in level_npixels]
        Ks = [1 + (2 * (1 + i // 2)) for i in range(depth + 1)]

        print(f"Channels: {base_channels}, K values: {Ks}")
        print(f"Depth: {depth}, NSides: {level_nsides}")

        inputs = tf.keras.Input(shape=self.input_shape[1:], name="lensed")

        # Build all layers for the unified network
        all_layers = []

        # Encoder path (downsampling)
        for i in range(depth):
            encoder = EncoderBlock(
                level_nsides[i],
                level_npixels[i],
                fin=base_channels[i],
                fout=base_channels[i + 1],
                activation=self.activation,
                max_batch_size=self.max_batch_size,
                K=Ks[i],
                dropout_rate=self.dropout_rate,
                use_bias=self.use_bias,
                use_bn=self.use_bn,
            )
            all_layers.extend(encoder.build_layers())
            all_layers.extend(encoder.get_pool_layer())

        # Bottleneck layer
        all_layers.append(
            HealpyChebyshev(
                K=Ks[-1],
                Fout=base_channels[-1],
                activation=self.activation,
                use_bias=self.use_bias,
                use_bn=self.use_bn,
            )
        )

        # Decoder path (upsampling)
        for i in reversed(range(1, depth + 1)):
            decoder = DecoderBlock(
                level_nsides[i],
                level_npixels[i],
                fin=base_channels[i],
                fout=base_channels[i],
                activation=self.activation,
                max_batch_size=self.max_batch_size,
                K=Ks[i],
                dropout_rate=self.dropout_rate,
                use_bias=self.use_bias,
                use_bn=self.use_bn,
            )
            all_layers.extend(decoder.build_layers())

        # Final decoder block
        final_decoder = DecoderBlock(
            level_nsides[0],
            level_npixels[0],
            fin=base_channels[1],
            fout=32,
            activation=self.activation,
            max_batch_size=self.max_batch_size,
            K=3,
            dropout_rate=0.1,
            use_bias=self.use_bias,
            use_bn=self.use_bn,
        )
        all_layers.extend(final_decoder.build_layers())

        # Output head layer
        all_layers.append(
            HealpyChebyshev(
                K=1,
                Fout=1,
                activation=None,
                use_bn=False,
                use_bias=True,
            )
        )

        # Create unified HealpyGCNN with all layers
        gcnn = HealpyGCNN(
            nside=self.nside,
            indices=np.arange(self.npix),
            layers=all_layers,
            n_neighbors=8,
            max_batch_size=self.max_batch_size,
            initial_Fin=self.npol,
            name="unified_unet",
        )

        outputs = gcnn(inputs)
        return tf.keras.Model(inputs, outputs)

In [ ]:
if not os.path.exists(unet_keras_file):
    logger.info("Creating U-Net model and training data")
    epoch_steps = math.ceil(
        core.total_sims * unet_split[0] * unet_duplicates[0] // batch_size
    )
    decay_steps = epoch_steps * 2

    ds = KappaDataset.fromCore(
        core,
        phi_scale=100,
        kappa_scale=np.sqrt(1e5),  # 1e7
        x_output="lensed",
        y_output="kappa",
    )

    train, val, test = ds.split(
        train_size=unet_split[0],
        val_size=unet_split[1],
        test_size=unet_split[2],
        to_tf=True,
        batch_size=batch_size,
        duplicates=unet_duplicates,
        buffer_size=len(ds),
        cache_file=unet_cache,
        gen_batch_size=core.slurm.n_cpus,
    )

    with strategy.scope():
        learning_rate = ExponentialDecay(initial_LR, decay_steps, 0.96, staircase=True)
        u_net = SimpleHealpyUNet(
            (None, core.npix, core.npols), activation="gelu", max_batch_size=batch_size
        ).get_model()
        u_net.compile(optimizer=Adam(learning_rate), loss="mse")

    u_net.summary()

    logger.info("Training U-Net")
    history = u_net.fit(
        train, epochs=30, validation_data=val, callbacks=callbacks, verbose=1
    )
    u_net.save(unet_keras_file)
    logger.info(f"U-Net saved to {unet_keras_file}")
else:
    logger.info(f"Loading U-Net from {unet_keras_file}")
    u_net = tf.keras.models.load_model(unet_keras_file)

In [ ]:
if "history" in locals():
    plt.figure(figsize=(10, 4))
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss (MSE)")
    plt.title("U-Net Training Loss")
    plt.yscale("log")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## U-Net Diagnostic Analysis
Analyze why the model may be failing to learn kappa prediction from lensed maps.

In [ ]:
# 1. Check input/output data statistics
print("=== Data Statistics ===")

# Get a sample batch from the dataset
for x_batch, y_batch in train.take(1):
    print(f"\nInput (lensed) shape: {x_batch.shape}")
    print(
        f"Input (lensed) - mean: {tf.reduce_mean(x_batch):.6f}, std: {tf.math.reduce_std(x_batch):.6f}"
    )
    print(
        f"Input (lensed) - min: {tf.reduce_min(x_batch):.6f}, max: {tf.reduce_max(x_batch):.6f}"
    )

    print(f"\nTarget (kappa) shape: {y_batch.shape}")
    print(
        f"Target (kappa) - mean: {tf.reduce_mean(y_batch):.6f}, std: {tf.math.reduce_std(y_batch):.6f}"
    )
    print(
        f"Target (kappa) - min: {tf.reduce_min(y_batch):.6f}, max: {tf.reduce_max(y_batch):.6f}"
    )

    # Check for NaN/Inf
    print(f"\nInput has NaN: {tf.reduce_any(tf.math.is_nan(x_batch))}")
    print(f"Input has Inf: {tf.reduce_any(tf.math.is_inf(x_batch))}")
    print(f"Target has NaN: {tf.reduce_any(tf.math.is_nan(y_batch))}")
    print(f"Target has Inf: {tf.reduce_any(tf.math.is_inf(y_batch))}")

In [ ]:
# 2. Visualize sample prediction vs ground truth
print("=== Sample Predictions ===")

for x_batch, y_batch in test.take(1):
    # Get predictions
    y_pred = u_net.predict(x_batch, verbose=0)

    # Select first sample
    idx = 0
    lensed_map = x_batch[idx, :, 0].numpy()
    true_kappa = y_batch[idx, :, 0].numpy()
    pred_kappa = y_pred[idx, :, 0]

    print(
        f"Prediction - mean: {np.mean(pred_kappa):.6f}, std: {np.std(pred_kappa):.6f}"
    )
    print(f"Prediction - min: {np.min(pred_kappa):.6f}, max: {np.max(pred_kappa):.6f}")
    print(
        f"True kappa - mean: {np.mean(true_kappa):.6f}, std: {np.std(true_kappa):.6f}"
    )

    # Check if model outputs constant values
    if np.std(pred_kappa) < 1e-6:
        print("\n⚠️ WARNING: Model outputs nearly constant values!")

    # Correlation
    corr = np.corrcoef(true_kappa.flatten(), pred_kappa.flatten())[0, 1]
    print(f"\nCorrelation (true vs pred): {corr:.4f}")

    # Plot
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    hp.mollview(lensed_map, title="Input (Lensed)", hold=True, sub=(1, 4, 1))
    hp.mollview(true_kappa, title="True Kappa", hold=True, sub=(1, 4, 2))
    hp.mollview(pred_kappa, title="Predicted Kappa", hold=True, sub=(1, 4, 3))
    hp.mollview(
        true_kappa - pred_kappa,
        title="Residual (True - Pred)",
        hold=True,
        sub=(1, 4, 4),
    )

    plt.tight_layout()
    plt.show()

    # Scatter plot
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.scatter(true_kappa.flatten(), pred_kappa.flatten(), alpha=0.1, s=1)
    ax.plot(
        [true_kappa.min(), true_kappa.max()],
        [true_kappa.min(), true_kappa.max()],
        "r--",
        label="Perfect",
    )
    ax.set_xlabel("True Kappa")
    ax.set_ylabel("Predicted Kappa")
    ax.set_title(f"Kappa Scatter (Corr: {corr:.4f})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# 3. Analyze gradient flow through the network
print("=== Gradient Flow Analysis ===")

for x_batch, y_batch in train.take(1):
    x_sample = x_batch[:1]  # Single sample
    y_sample = y_batch[:1]

    with tf.GradientTape() as tape:
        tape.watch(x_sample)
        y_pred = u_net(x_sample, training=True)
        loss = tf.reduce_mean(tf.square(y_pred - y_sample))

    # Get gradients w.r.t input
    input_grad = tape.gradient(loss, x_sample)

    print(f"Loss value: {loss.numpy():.6f}")
    print(f"Input gradient - mean: {tf.reduce_mean(tf.abs(input_grad)):.6e}")
    print(f"Input gradient - max: {tf.reduce_max(tf.abs(input_grad)):.6e}")

    if tf.reduce_mean(tf.abs(input_grad)) < 1e-10:
        print("\n⚠️ WARNING: Vanishing gradients detected!")

    # Check layer weights and gradients
    print("\n=== Layer-wise Analysis ===")
    with tf.GradientTape() as tape:
        y_pred = u_net(x_sample, training=True)
        loss = tf.reduce_mean(tf.square(y_pred - y_sample))

    grads = tape.gradient(loss, u_net.trainable_variables)

    for i, (var, grad) in enumerate(zip(u_net.trainable_variables, grads)):
        if grad is not None:
            grad_mean = tf.reduce_mean(tf.abs(grad)).numpy()
            weight_mean = tf.reduce_mean(tf.abs(var)).numpy()
            print(f"{var.name[:50]:50s} | W: {weight_mean:.2e} | G: {grad_mean:.2e}")
        else:
            print(f"{var.name[:50]:50s} | Gradient is None!")

In [ ]:
# 4. Check layer output shapes through the unified HealpyGCNN
print("=== Layer Architecture Analysis ===")

# The unified HealpyGCNN is the first layer after input
gcnn_layer = u_net.layers[1]  # unified_unet layer
print(f"GCNN Layer: {gcnn_layer.name}")
print(f"GCNN nside: {gcnn_layer.nside}")
print(f"GCNN npix (from indices): {len(gcnn_layer.indices)}")

# List all internal layers
print(f"\nInternal layers in HealpyGCNN ({len(gcnn_layer.layers)}):")
for i, layer in enumerate(gcnn_layer.layers):
    layer_type = type(layer).__name__
    if hasattr(layer, "Fout"):
        print(f"  {i:2d}. {layer_type:30s} | Fout={layer.Fout}")
    elif hasattr(layer, "p"):
        print(f"  {i:2d}. {layer_type:30s} | p={layer.p} (pooling factor)")
    else:
        print(f"  {i:2d}. {layer_type:30s}")

In [ ]:
# 5. Compare with working neo_trainer architecture pattern
print("=== Architecture Comparison: slim vs neo ===")
print("\nKey Difference: Unified vs Separate HealpyGCNN")
print("-" * 60)

print(
    """
SLIM_TRAINER (Current - Broken):
  - Single HealpyGCNN with nside={nside} for ALL layers
  - Pool/Upsample layers inside same graph structure
  - Graph indices always = np.arange({npix})
  
  PROBLEM: HealpyGCNN graph is resolution-specific!
  When HealpyPool reduces resolution, the graph structure 
  (Laplacian, adjacency) doesn't update - it still uses 
  full-resolution connectivity.

NEO_TRAINER (Working):
  - Separate HealpyGCNN per resolution level
  - Each block creates graph at correct nside/npix
  - Pooling layer in its own HealpyGCNN context
  
  WHY IT WORKS: Each HealpyGCNN builds the correct graph
  for its resolution level.
""".format(
        nside=core.nside, npix=core.npix
    )
)

print("\n=== Resolution Tracking Through Network ===")
depth = 3
level_npixels = [core.npix // (4**i) for i in range(depth + 1)]
level_nsides = [hp.npix2nside(npix) for npix in level_npixels]

print(f"Expected resolutions at each depth level:")
for i, (ns, np_) in enumerate(zip(level_nsides, level_npixels)):
    print(f"  Level {i}: nside={ns}, npix={np_}")

print(f"\nActual HealpyGCNN configuration:")
print(f"  nside={gcnn_layer.nside}, npix={len(gcnn_layer.indices)}")
print(f"\n⚠️ The graph is FIXED at full resolution even though")
print(f"   pool layers expect changing resolutions!")

In [ ]:
# 6. Test intermediate tensor shapes with debug forward pass
print("=== Debug Forward Pass ===")

# Create a simple test input
test_input = tf.random.normal([1, core.npix, core.npols])
print(f"Test input shape: {test_input.shape}")

# Get the HealpyGCNN layer
gcnn_layer = u_net.layers[1]

# Try to trace through internal layers manually
# Note: This may not work directly due to HealpyGCNN internals
try:
    output = u_net(test_input, training=False)
    print(f"Output shape: {output.shape}")
    print(
        f"Output stats - mean: {tf.reduce_mean(output):.6f}, std: {tf.math.reduce_std(output):.6f}"
    )

    # Expected: output should have shape [1, npix, 1] with meaningful values
    if output.shape[1] != core.npix:
        print(
            f"\n⚠️ WARNING: Output npix ({output.shape[1]}) != input npix ({core.npix})"
        )
        print("   This indicates resolution mismatch in pool/upsample operations!")

except Exception as e:
    print(f"Error during forward pass: {e}")

In [ ]:
# 7. Training loss analysis
print("=== Training Loss Analysis ===")

if "history" in locals():
    initial_loss = history.history["loss"][0]
    final_loss = history.history["loss"][-1]
    min_loss = min(history.history["loss"])

    print(f"Initial loss: {initial_loss:.6f}")
    print(f"Final loss: {final_loss:.6f}")
    print(f"Minimum loss: {min_loss:.6f}")
    print(f"Loss reduction: {(initial_loss - final_loss) / initial_loss * 100:.1f}%")

    # Check if loss plateaued immediately
    loss_change_epoch1 = abs(history.history["loss"][1] - history.history["loss"][0])
    avg_loss_change = np.mean(np.abs(np.diff(history.history["loss"])))

    print(f"\nLoss change after epoch 1: {loss_change_epoch1:.6f}")
    print(f"Average loss change per epoch: {avg_loss_change:.6f}")

    if loss_change_epoch1 < 1e-6:
        print("\n⚠️ Loss barely changed from start - likely architectural issue")
    elif final_loss > initial_loss * 0.9:
        print("\n⚠️ Loss reduced by <10% - model struggling to learn")
    else:
        print("\n✓ Loss is decreasing - check prediction quality")
else:
    print("No training history available - run training first")

### Diagnostic Summary & Recommendations

Based on the analysis above:

1. **If model outputs constant/near-constant values**: The unified HealpyGCNN approach has broken the resolution handling. Pool/Upsample layers operate on incorrect graph structures.

2. **If gradients are vanishing**: Skip connections are critical for this inverse problem. Without them, gradient signal cannot flow back effectively through the deep encoder-decoder.

3. **If loss plateaus immediately**: Architectural issue - the network cannot learn the mapping.

**Recommended Fixes:**
- **Option A**: Restore separate HealpyGCNN per resolution (like neo_trainer) but without skip connections
- **Option B**: Remove pooling/upsampling entirely and use a fixed-resolution network
- **Option C**: Add skip connections back (defeats purpose of "slim" but may be necessary)

## Delens lensed maps and train fnl model

In [ ]:
res_arcmin = hp.nside2resol(core.nside, arcmin=True)
res_rad = res_arcmin * utils.arcmin
ny = int(round(np.pi / res_rad))
res_fixed = np.pi / ny
shape, wcs = enmap.fullsky_geometry(res_fixed)


def _delens_py(lensed, kappa, nside=core.nside):
    """Delens maps using kappa predictions by converting to phi"""
    delensed_maps = np.zeros_like(lensed)
    for i, (l_map, k_map) in enumerate(zip(lensed, kappa)):
        l_map = hp.reorder(l_map.numpy().flatten(), n2r=True).astype(np.float32)
        k_map = hp.reorder(k_map.numpy().flatten(), n2r=True).astype(np.float32)

        alm_kappa = hp.map2alm(k_map, lmax=3 * core.nside - 1)
        lmax_kappa = hp.Alm.getlmax(len(alm_kappa))
        inv_kappa_coeff = -2.0 / (
            np.arange(lmax_kappa, dtype=np.float64)
            * np.arange(1, lmax_kappa + 1, dtype=np.float64)
        )
        inv_kappa_coeff[0] = 0
        alm_phi = hp.almxfl(alm_kappa, inv_kappa_coeff)
        p_map_phi = hp.alm2map(alm_phi, core.nside * 2)

        lensed_enmap = reproject.healpix2map(l_map, shape, wcs)
        phi_enmap = reproject.healpix2map(p_map_phi, shape, wcs)
        delensed_enmap = lensing.delens_map(lensed_enmap, phi_enmap)

        m = reproject.map2healpix(delensed_enmap, nside)
        m = hp.remove_dipole(m, copy=False)
        delensed_maps[i] = m[:, None]

    return delensed_maps


def _map_fn(pair, kappa):
    lensed, fnls = pair
    delensed = tf.py_function(func=_delens_py, inp=[lensed, kappa], Tout=tf.float32)
    delensed.set_shape([None, core.npix, core.npols])
    fnls.set_shape([None, len(shapes)])
    return delensed, fnls

In [ ]:
logger.info("Creating delensed dataset for fnl training")
ds_lensed = MapDataset.fromCore(core, lensed=True)
train_3, val_3, test_3 = ds_lensed.split(
    train_size=final_split[0],
    val_size=final_split[1],
    test_size=final_split[2],
    to_tf=True,
    batch_size=batch_size,
    duplicates=final_duplicates,
    gen_batch_size=core.slurm.n_cpus,
)

fnl_truth = np.concatenate([y for _, y in test_3])

# Get kappa predictions and delens
logger.info("Predicting kappa with U-Net and delensing maps")
kappa_preds = u_net.predict(train_3, verbose=0)
kappa_preds_delens = ds.kappa_to_phi(kappa_preds)
kappa_ds = tf.data.Dataset.from_tensor_slices(kappa_preds_delens).batch(batch_size)

delensed_train = (
    tf.data.Dataset.zip((train_3, kappa_ds))
    .map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

# Same for validation
kappa_preds_val = u_net.predict(val_3, verbose=0)
kappa_preds_val_delens = ds.kappa_to_phi(kappa_preds_val)
kappa_ds_val = tf.data.Dataset.from_tensor_slices(kappa_preds_val_delens).batch(
    batch_size
)
delensed_val = (
    tf.data.Dataset.zip((val_3, kappa_ds_val))
    .map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

# And for test
kappa_preds_test = u_net.predict(test_3, verbose=0)
kappa_preds_test_delens = ds.kappa_to_phi(kappa_preds_test)
kappa_ds_test = tf.data.Dataset.from_tensor_slices(kappa_preds_test_delens).batch(
    batch_size
)
delensed_test = (
    tf.data.Dataset.zip((test_3, kappa_ds_test))
    .map(_map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
logger.info("Creating and training fnl model on delensed maps")
epoch_steps = math.ceil(
    core.total_sims * final_split[0] * final_duplicates[0] // batch_size
)
decay_steps = epoch_steps * 3

with strategy.scope():
    learning_rate = ExponentialDecay(initial_LR, decay_steps, 0.95, staircase=True)
    fnl_model = get_fnl_model((None, core.npix, core.npols), batch_size, len(shapes))
    fnl_model.compile(
        optimizer=AdamW(learning_rate), loss=RMSELoss(), metrics=rmse_metrics(shapes)
    )

logger.info("Training fnl model")
fnl_history = fnl_model.fit(
    delensed_train,
    epochs=max_epochs,
    validation_data=delensed_val,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(fnl_history.history["loss"], label="Train Loss")
plt.plot(fnl_history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("fnl Model Training Loss")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## fnl Predictions on Delensed Test Set

In [ ]:
logger.info("Predicting fnl on delensed test set")
fnl_preds = fnl_model.predict(delensed_test, verbose=0)

fnl_truth_flat = fnl_truth.ravel()
fnl_preds_flat = fnl_preds.ravel()

print(f"True fnls shape: {fnl_truth_flat.shape}")
print(f"Predicted fnls shape: {fnl_preds_flat.shape}")

fnl_error = fnl_preds_flat - fnl_truth_flat
print(f"\nMetrics:")
print(f"Mean absolute error: {np.mean(np.abs(fnl_error)):.4f}")
print(f"RMSE: {np.sqrt(np.mean(fnl_error**2)):.4f}")

sigma = core.get_likelihoods(True)[0]
print(f"Sigma (likelihood bound): {sigma:.4f}")

In [ ]:
line = np.array([np.nanmin(fnl_truth_flat), np.nanmax(fnl_truth_flat)])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Scatter plot
axes[0].scatter(fnl_truth_flat, fnl_preds_flat, alpha=0.5, s=10)
axes[0].plot(line, line, "r--", label="Perfect prediction")
axes[0].plot(line, line + sigma, "g--", label=f"+σ={sigma:.1f}")
axes[0].plot(line, line - sigma, "g--", label=f"-σ={-sigma:.1f}")
axes[0].set_xlabel("True fnl")
axes[0].set_ylabel("Predicted fnl")
axes[0].set_title("fnl Prediction from Delensed Maps")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error histogram
axes[1].hist(fnl_error, bins=50, color="C0", alpha=0.8, edgecolor="black")
axes[1].axvline(sigma, color="g", linestyle="--", linewidth=2, label=f"+σ={sigma:.1f}")
axes[1].axvline(
    -sigma, color="g", linestyle="--", linewidth=2, label=f"-σ={-sigma:.1f}"
)
axes[1].set_xlabel("Prediction Error")
axes[1].set_ylabel("Count")
axes[1].set_title("fnl Error Distribution")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()